# Section 1: Introduction

Notebook này gom toàn bộ quy trình cho đề tài phân loại ảnh phương tiện giao thông: mô tả dataset, làm sạch dữ liệu, chia train/validation/test, tiền xử lý ảnh, trực quan hóa đặc trưng, train MobileNet và ResNet, đánh giá và thảo luận kết quả.

**Danh sách class mục tiêu (phiên bản hiện tại):** `bicycle`, `boat`, `bus`, `car`, `helicopter`, `minibus`, `motorcycle`, `train`, `truck`.

**Mục tiêu dữ liệu:** hơn 10.000 ảnh sau khi crawl và làm sạch, mỗi class có số lượng mẫu tương đối đủ để tránh mất cân bằng quá lớn.

**Cấu hình mô hình trong notebook:** sử dụng MobileNetV2 và ResNet50 từ Keras Applications, thay classifier cuối bằng head phân loại 9 lớp và huấn luyện theo 2 giai đoạn: warm-up classifier head, sau đó mở một phần backbone với learning rate nhỏ.



## Section 2: Environment Setup

Cell dưới đây import thư viện, kiểm tra GPU, cấu hình đường dẫn dataset và các siêu tham số huấn luyện. Notebook được thiết kế để chạy linh hoạt trên local, Google Colab và Kaggle.

Với Kaggle, bạn có thể upload dataset theo một trong các dạng sau:

- `data/cleaned/<class_name>/*.jpg`
- `cleaned/<class_name>/*.jpg`
- `<class_name>/*.jpg` nếu file zip chứa trực tiếp các folder class
- `data/splits/train|val|test/<class_name>/*.jpg` nếu dataset đã chia sẵn
- một file `.zip` chứa `data/cleaned` hoặc `cleaned`

Notebook ưu tiên đọc dataset đã chia sẵn trong `data/splits`; nếu không có thì đọc `data/cleaned` và tự split stratified.

Notebook được tối ưu để chạy trên Kaggle GPU T4x2: tự nhận nhiều GPU bằng `tf.distribute.MirroredStrategy` và tự tăng global batch size theo số GPU khả dụng. Mặc định dùng `float32` để ưu tiên ổn định số học khi huấn luyện 2 giai đoạn.



In [ ]:
from pathlib import Path
import os
import json
import math
import random
import shutil
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

try:
    import imagehash
except ImportError:
    imagehash = None
    print('imagehash is not installed. pHash/dHash demo cells will be skipped unless you install it.')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

REQUIRED_CLASSES = ['bicycle', 'boat', 'bus', 'car', 'helicopter', 'minibus', 'motorcycle', 'train', 'truck']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_NAMES = ['train', 'val', 'valid', 'validation', 'test']
CONTAINER_DIR_NAMES = {'data', 'cleaned', 'raw', 'splits', 'train', 'val', 'valid', 'validation', 'test', 'images'}

IMG_SIZE = (224, 224)
PER_REPLICA_BATCH_SIZE = 32  # T4x2: global batch = 32 * 2 = 64

# Two-phase training schedule
USE_PRETRAINED = True
WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = 30
EPOCHS = WARMUP_EPOCHS + FINETUNE_EPOCHS
WARMUP_LR = 1e-3
FINETUNE_LR = 1e-5
LEARNING_RATE = WARMUP_LR
MOBILENET_UNFREEZE_LAYERS = 30
RESNET_UNFREEZE_LAYERS = 60

MOBILENET_DROPOUT = 0.35
RESNET_DROPOUT = 0.35
USE_MIXUP = True
MIXUP_ALPHA = 0.15  # nhẹ, nằm trong khoảng 0.1-0.2
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
FOCAL_LABEL_SMOOTHING = 0.0
USE_CLASS_WEIGHTS = False  # chỉ bật khi cần, đã có cap để tránh weight quá lớn
CLASS_WEIGHT_CAP = 2.0
CLASS_WEIGHT_FLOOR = 0.5
EARLY_STOPPING_PATIENCE = 10
REDUCE_LR_PATIENCE = 4
REDUCE_LR_FACTOR = 0.3
MIN_LR = 1e-6
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Kaggle T4x2 optimization: use both GPUs when available.
GPU_DEVICES = tf.config.list_physical_devices('GPU')
for gpu in GPU_DEVICES:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print('Cannot set GPU memory growth after initialization:', exc)

if len(GPU_DEVICES) > 1:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.get_strategy()

NUM_REPLICAS = strategy.num_replicas_in_sync
BATCH_SIZE = PER_REPLICA_BATCH_SIZE * NUM_REPLICAS

# Keep float32 to reduce NaN risk on Kaggle when using augmentation + MixUp + focal loss.
USE_MIXED_PRECISION = False
tf.keras.mixed_precision.set_global_policy('float32')

RUN_TRAINING = True
EXPORT_SPLIT_FOLDERS = False  # Set True to export data/splits folders from this notebook
MAX_IMAGES_PER_CLASS = None   # Set a small number, e.g. 200, for a quick smoke test


def contains_images(path):
    path = Path(path)
    if not path.exists():
        return False
    return any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob('*'))


def has_class_folders(path):
    """Check whether a folder has the structure <root>/<class_name>/*.jpg."""
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False

    class_dirs = [child for child in path.iterdir() if child.is_dir() and child.name.lower() not in CONTAINER_DIR_NAMES]
    image_class_dirs = [child for child in class_dirs if contains_images(child)]

    if not image_class_dirs:
        return False

    names = {child.name for child in image_class_dirs}
    has_required_class = bool(names.intersection(REQUIRED_CLASSES))

    # This project should have multiple classes; avoid treating container folders as class roots.
    return has_required_class or len(image_class_dirs) >= 2


def has_split_folders(path):
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False
    names = {child.name.lower() for child in path.iterdir() if child.is_dir()}
    has_split_names = 'train' in names and (('val' in names) or ('valid' in names) or ('validation' in names)) and 'test' in names
    return has_split_names and contains_images(path)


def add_dataset_candidates(base, cleaned_candidates, split_candidates):
    """Add common dataset structures to candidate lists."""
    base = Path(base)
    split_candidates.extend([
        base / 'data' / 'splits',
        base / 'splits',
    ])
    cleaned_candidates.extend([
        base / 'data' / 'cleaned',
        base / 'cleaned',
        base,
    ])


def extract_zip_if_needed(zip_path, extract_base):
    """Extract a Kaggle zip dataset into /kaggle/working if needed."""
    zip_path = Path(zip_path)
    extract_base = Path(extract_base)
    extract_dir = extract_base / zip_path.stem
    marker = extract_dir / '.extracted_from_zip'

    if marker.exists() and contains_images(extract_dir):
        return extract_dir

    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f'Extracting zip dataset: {zip_path} -> {extract_dir}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

    marker.write_text('ok', encoding='utf-8')
    return extract_dir


def find_dataset_paths():
    """Detect dataset root for local/Colab/Kaggle with robust fallbacks."""
    cwd = Path.cwd()
    cleaned_candidates = []
    split_candidates = []

    # Current project and parent fallbacks
    add_dataset_candidates(cwd, cleaned_candidates, split_candidates)
    if cwd.parent != cwd:
        add_dataset_candidates(cwd.parent, cleaned_candidates, split_candidates)

    # Colab common paths
    colab_bases = [Path('/content'), Path('/content/drive/MyDrive')]
    for base in colab_bases:
        add_dataset_candidates(base, cleaned_candidates, split_candidates)

    # Kaggle common paths
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        add_dataset_candidates(kaggle_input, cleaned_candidates, split_candidates)
        # Search one level and two levels deep for datasets
        for level1 in kaggle_input.iterdir():
            if level1.is_dir():
                add_dataset_candidates(level1, cleaned_candidates, split_candidates)
                for level2 in level1.iterdir():
                    if level2.is_dir():
                        add_dataset_candidates(level2, cleaned_candidates, split_candidates)

        # If Kaggle upload is a zip, extract and add extracted folder as candidate
        extracted_base = Path('/kaggle/working/extracted_datasets')
        extracted_base.mkdir(parents=True, exist_ok=True)
        zip_candidates = list(kaggle_input.rglob('*.zip'))
        for zip_path in zip_candidates:
            try:
                extracted_dir = extract_zip_if_needed(zip_path, extracted_base)
                add_dataset_candidates(extracted_dir, cleaned_candidates, split_candidates)
            except Exception as exc:
                print(f'Skip zip {zip_path} due to extraction error: {exc}')

    # Deduplicate while preserving order
    def dedupe(paths):
        seen = set()
        result = []
        for p in paths:
            p = p.resolve() if p.exists() else p
            if p not in seen:
                seen.add(p)
                result.append(p)
        return result

    split_candidates = dedupe(split_candidates)
    cleaned_candidates = dedupe(cleaned_candidates)

    # Prefer already-split dataset if available
    for candidate in split_candidates:
        if has_split_folders(candidate):
            return candidate, 'splits'

    # Then fallback to cleaned dataset
    for candidate in cleaned_candidates:
        if has_class_folders(candidate):
            return candidate, 'cleaned'

    raise FileNotFoundError(
        'Cannot find dataset folder. Please set DATA_ROOT manually. '
        'Expected cleaned structure <root>/<class_name>/*.jpg '
        'or split structure <root>/train|val|test/<class_name>/*.jpg.'
    )




In [ ]:
def print_kaggle_input_tree(max_lines=80):
    root = Path('/kaggle/input')
    if not root.exists():
        return
    print('Available Kaggle input folders:')
    shown = 0
    for path in sorted(root.rglob('*')):
        if shown >= max_lines:
            print(f'... skipped after {max_lines} entries')
            break
        depth = len(path.relative_to(root).parts)
        if depth <= 4:
            print(path)
            shown += 1


# Ưu tiên các đường dẫn Kaggle phổ biến trước. Nếu không thấy mới fallback sang auto-search.
DATA_ROOT = None
DATA_MODE = None
KAGGLE_DATA_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/leighk/vehicle-dataset/cleaned'),
    Path('/kaggle/input/datasets/leighk/vehicle-dataset/data/cleaned'),
    Path('/kaggle/input/vehicle-dataset/cleaned'),
    Path('/kaggle/input/vehicle-dataset/data/cleaned'),
    Path('/kaggle/input/vehicle-dataset'),
    Path('/kaggle/input/traffic-vehicle-cleaned/cleaned'),
    Path('/kaggle/input/traffic-vehicle-cleaned/data/cleaned'),
]

for candidate in KAGGLE_DATA_ROOT_CANDIDATES:
    if candidate.exists() and has_class_folders(candidate):
        DATA_ROOT = candidate
        DATA_MODE = 'cleaned'
        break
    if candidate.exists() and has_split_folders(candidate):
        DATA_ROOT = candidate
        DATA_MODE = 'splits'
        break

if DATA_ROOT is None:
    try:
        DATA_ROOT, DATA_MODE = find_dataset_paths()
    except FileNotFoundError:
        print_kaggle_input_tree()
        raise

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Store trained models, histories, test predictions, and report figures.
ARTIFACT_DIR = OUTPUT_DIR / 'training_artifacts'
MODEL_DIR = ARTIFACT_DIR / 'models'
FIGURE_DIR = ARTIFACT_DIR / 'figures'

for directory in [ARTIFACT_DIR, MODEL_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Distribution strategy:', type(strategy).__name__)
print('Number of replicas:', NUM_REPLICAS)
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
print('Per-replica batch size:', PER_REPLICA_BATCH_SIZE)
print('Global batch size:', BATCH_SIZE)
print('USE_PRETRAINED:', USE_PRETRAINED)
print('Warmup/Fine-tune epochs:', WARMUP_EPOCHS, '/', FINETUNE_EPOCHS, '| total =', EPOCHS)
print('Warmup/Fine-tune LR:', WARMUP_LR, '/', FINETUNE_LR)
print('Use MixUp:', USE_MIXUP, '| alpha =', MIXUP_ALPHA)
print('Use focal loss:', USE_FOCAL_LOSS, '| gamma =', FOCAL_GAMMA)
print('Use class weights:', USE_CLASS_WEIGHTS, '| cap =', CLASS_WEIGHT_CAP)
print('DATA_MODE:', DATA_MODE)
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)




## Section 3: Dataset Loading

Notebook đọc ảnh từ folder dataset đã tìm được ở Section 2, tự lấy tên class từ tên thư mục và tạo dataframe gồm `image_path`, `label`, `width`, `height`, `channel`, `file_size`.

Cách khuyến nghị khi train trên Kaggle là upload file zip chứa cấu trúc `data/cleaned/<class_name>/*.jpg`. Nếu Kaggle giữ nguyên file `.zip`, notebook sẽ tự giải nén vào `/kaggle/working/extracted_datasets/` rồi đọc dữ liệu từ đó.




In [ ]:
def get_image_info(path):
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img)
            width, height = img.size
            channel = len(img.getbands())
            return {
                'width': width,
                'height': height,
                'channel': channel,
                'mode': img.mode,
                'format': img.format,
                'file_size': path.stat().st_size,
                'file_size_kb': path.stat().st_size / 1024,
                'error': None,
            }
    except Exception as exc:
        return {
            'width': np.nan,
            'height': np.nan,
            'channel': np.nan,
            'mode': None,
            'format': None,
            'file_size': path.stat().st_size if path.exists() else np.nan,
            'file_size_kb': path.stat().st_size / 1024 if path.exists() else np.nan,
            'error': str(exc),
        }


def normalized_split_name(name):
    name = name.lower()
    if name in {'val', 'valid', 'validation'}:
        return 'val'
    return name


def scan_cleaned_dataset(root):
    class_names = sorted([p.name for p in root.iterdir() if p.is_dir()])
    rows = []
    for label in class_names:
        image_paths = [p for p in sorted((root / label).rglob('*')) if p.suffix.lower() in IMAGE_EXTENSIONS]
        if MAX_IMAGES_PER_CLASS is not None:
            image_paths = image_paths[:MAX_IMAGES_PER_CLASS]
        for path in image_paths:
            rows.append({'image_path': str(path), 'label': label, 'split': None, **get_image_info(path)})
    return class_names, rows


def scan_split_dataset(root):
    rows = []
    class_names = set()
    for split_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        split_name = normalized_split_name(split_dir.name)
        if split_name not in {'train', 'val', 'test'}:
            continue
        for label_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            label = label_dir.name
            class_names.add(label)
            image_paths = [p for p in sorted(label_dir.rglob('*')) if p.suffix.lower() in IMAGE_EXTENSIONS]
            if MAX_IMAGES_PER_CLASS is not None:
                image_paths = image_paths[:MAX_IMAGES_PER_CLASS]
            for path in image_paths:
                rows.append({'image_path': str(path), 'label': label, 'split': split_name, **get_image_info(path)})
    return sorted(class_names), rows


if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DATA_ROOT}. Please update DATA_ROOT in Section 2.')

if DATA_MODE == 'splits':
    class_names, rows = scan_split_dataset(DATA_ROOT)
else:
    class_names, rows = scan_cleaned_dataset(DATA_ROOT)

print('Detected classes:', class_names)
missing_required = sorted(set(REQUIRED_CLASSES) - set(class_names))
if missing_required:
    print('Warning - missing required classes:', missing_required)

df_all = pd.DataFrame(rows)
if df_all.empty:
    raise ValueError('No images found. Check DATA_ROOT and folder structure.')

error_df = df_all[df_all['error'].notna()].copy()
df = df_all[df_all['error'].isna()].copy().reset_index(drop=True)
df['aspect_ratio'] = df['width'] / df['height']

print('Data mode:', DATA_MODE)
print('Total files scanned:', len(df_all))
print('Valid images:', len(df))
print('Unreadable/error images:', len(error_df))
if DATA_MODE == 'splits':
    display(df.groupby(['split', 'label']).size().unstack(fill_value=0))
display(df.head())



## Section 4: Dataset Overview and Statistics

Phần này thống kê tổng số mẫu, số mẫu từng class, kiểu dữ liệu, số mẫu lỗi/trống, kích thước ảnh và trực quan hóa bằng bar chart, histogram, boxplot.



In [ ]:
print('Total valid samples:', len(df))
print('Number of classes:', df['label'].nunique())

class_counts = df['label'].value_counts().sort_index()
display(class_counts.to_frame('count'))

print('Data types:')
display(df.dtypes.to_frame('dtype'))

print('Missing values per column:')
display(df.isna().sum().to_frame('missing_count'))

size_summary = df[['width', 'height', 'channel', 'file_size_kb', 'aspect_ratio']].describe().T
display(size_summary.round(2))

label_size_summary = df.groupby('label').agg(
    count=('image_path', 'count'),
    width_min=('width', 'min'), width_max=('width', 'max'), width_mean=('width', 'mean'), width_median=('width', 'median'),
    height_min=('height', 'min'), height_max=('height', 'max'), height_mean=('height', 'mean'), height_median=('height', 'median'),
    file_size_kb_mean=('file_size_kb', 'mean'),
).reset_index()
display(label_size_summary.round(2))



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=df, x='label', order=class_counts.index, ax=axes[0, 0])
axes[0, 0].set_title('Class distribution')
axes[0, 0].tick_params(axis='x', rotation=35)

sns.histplot(data=df, x='width', bins=40, kde=True, ax=axes[0, 1], color='#2f6f9f')
axes[0, 1].set_title('Width histogram')

sns.histplot(data=df, x='height', bins=40, kde=True, ax=axes[1, 0], color='#3c8d57')
axes[1, 0].set_title('Height histogram')

sns.boxplot(data=df, x='label', y='file_size_kb', order=class_counts.index, ax=axes[1, 1])
axes[1, 1].set_title('File size boxplot by class')
axes[1, 1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=df, x='label', y='width', order=class_counts.index, ax=axes[0])
axes[0].set_title('Width boxplot by class')
axes[0].tick_params(axis='x', rotation=35)

sns.boxplot(data=df, x='label', y='height', order=class_counts.index, ax=axes[1])
axes[1].set_title('Height boxplot by class')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()



In [ ]:
def show_samples(dataframe, labels, samples_per_class=3):
    sample_rows = []
    for label in labels:
        label_df = dataframe[dataframe['label'] == label]
        if len(label_df) == 0:
            continue
        sample_rows.extend(label_df.sample(min(samples_per_class, len(label_df)), random_state=SEED).to_dict('records'))

    cols = samples_per_class
    rows_n = max(1, int(np.ceil(len(sample_rows) / cols)))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 3 * rows_n))
    axes = np.array(axes).reshape(-1)
    for ax, row in zip(axes, sample_rows):
        img = Image.open(row['image_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(row['label'])
        ax.axis('off')
    for ax in axes[len(sample_rows):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(df, class_counts.index.tolist(), samples_per_class=3)



## Section 5: Data Cleaning Summary

Quy trình làm sạch dữ liệu nên được thực hiện sau khi crawl:

1. Mở từng ảnh bằng Pillow/OpenCV để phát hiện ảnh lỗi hoặc ảnh không đọc được.
2. Loại ảnh quá nhỏ, ví dụ nhỏ hơn `128x128`, vì ảnh quá nhỏ thường thiếu chi tiết để phân loại.
3. Chuẩn hóa ảnh về RGB JPEG để giảm lỗi khi load dataset.
4. Đổi tên ảnh theo format thống nhất, ví dụ `car_000001.jpg`.
5. Lọc ảnh trùng hoặc gần trùng bằng pHash/dHash. Hai ảnh có hash gần nhau theo Hamming distance nhỏ hơn ngưỡng, ví dụ `<= 6`, được xem là trùng hoặc gần trùng.

Công thức Hamming distance: số bit khác nhau giữa hai chuỗi hash. Với `imagehash`, có thể tính trực tiếp bằng phép trừ: `distance = hash_1 - hash_2`.



In [ ]:
def count_folder_images(root):
    root = Path(root)
    rows = []
    if not root.exists():
        return pd.DataFrame(columns=['label', 'count'])
    for label_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        count = sum(1 for p in label_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
        rows.append({'label': label_dir.name, 'count': count})
    return pd.DataFrame(rows)

raw_root = DATA_ROOT.parent / 'raw'
cleaned_root = DATA_ROOT
rejected_root = DATA_ROOT.parent / 'rejected'
duplicates_root = DATA_ROOT.parent / 'duplicates'

raw_counts = count_folder_images(raw_root).rename(columns={'count': 'raw_count'})
clean_counts = count_folder_images(cleaned_root).rename(columns={'count': 'cleaned_count'})
rejected_counts = count_folder_images(rejected_root).rename(columns={'count': 'rejected_count'})
duplicate_counts = count_folder_images(duplicates_root).rename(columns={'count': 'duplicate_count'})

cleaning_summary = clean_counts.merge(raw_counts, on='label', how='left')
cleaning_summary = cleaning_summary.merge(rejected_counts, on='label', how='left')
cleaning_summary = cleaning_summary.merge(duplicate_counts, on='label', how='left')
cleaning_summary = cleaning_summary.fillna(0)
display(cleaning_summary)

print('If raw/rejected/duplicates folders are unavailable on Kaggle, the table still reports cleaned_count from DATA_ROOT.')



In [ ]:
def compute_image_hash(path, method='phash'):
    if imagehash is None:
        return None
    with Image.open(path) as img:
        if method == 'phash':
            return imagehash.phash(img)
        if method == 'dhash':
            return imagehash.dhash(img)
        raise ValueError('method must be phash or dhash')

sample_paths = df['image_path'].head(2).tolist()
if imagehash is not None and len(sample_paths) == 2:
    h1 = compute_image_hash(sample_paths[0], 'phash')
    h2 = compute_image_hash(sample_paths[1], 'phash')
    print('pHash image 1:', h1)
    print('pHash image 2:', h2)
    print('Hamming distance:', h1 - h2)
else:
    print('Skip hash demo because imagehash is not installed or not enough images.')



In [ ]:
def show_rejected_or_duplicate_examples(root, title, max_images=8):
    root = Path(root)
    paths = []
    if root.exists():
        paths = [p for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS][:max_images]
    if not paths:
        print(f'No example images found for {title}: {root}')
        return
    cols = min(4, len(paths))
    rows_n = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 3 * rows_n))
    axes = np.array(axes).reshape(-1)
    for ax, path in zip(axes, paths):
        ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(path.parent.name)
        ax.axis('off')
    for ax in axes[len(paths):]:
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_rejected_or_duplicate_examples(rejected_root, 'Rejected images')
show_rejected_or_duplicate_examples(duplicates_root, 'Duplicate images')



## Section 6: Train / Validation / Test Split

Nếu dataset đang ở dạng `data/cleaned/<class_name>/`, notebook tự chia theo tỉ lệ `70/15/15` bằng stratified split theo label. Đây là flow khuyến nghị khi bạn upload file zip `data/cleaned` lên Kaggle.

Nếu dataset đã có sẵn `data/splits/train|val|test/<class_name>/`, notebook sẽ dùng split có sẵn và chỉ kiểm tra phân bố class giữa các tập.




In [ ]:
if DATA_MODE == 'splits' and df['split'].notna().all():
    split_df = df.copy()
    split_df['split'] = split_df['split'].map(normalized_split_name)
    required_splits = {'train', 'val', 'test'}
    existing_splits = set(split_df['split'].unique())
    missing_splits = required_splits - existing_splits
    if missing_splits:
        raise ValueError(f'Pre-split dataset is missing split folders: {missing_splits}')

    train_df = split_df[split_df['split'] == 'train'].copy()
    val_df = split_df[split_df['split'] == 'val'].copy()
    test_df = split_df[split_df['split'] == 'test'].copy()
    print('Using existing split folders from DATA_ROOT.')
else:
    if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-6:
        raise ValueError('TRAIN_RATIO + VAL_RATIO + TEST_RATIO must equal 1.0')

    min_class_count = df['label'].value_counts().min()
    if min_class_count < 3:
        raise ValueError('Each class needs at least 3 images for train/val/test stratified split.')

    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_RATIO,
        stratify=df['label'],
        random_state=SEED,
    )
    val_relative_ratio = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_relative_ratio,
        stratify=train_val_df['label'],
        random_state=SEED,
    )

    train_df = train_df.copy(); train_df['split'] = 'train'
    val_df = val_df.copy(); val_df['split'] = 'val'
    test_df = test_df.copy(); test_df['split'] = 'test'
    split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    print('Created stratified split from cleaned dataset.')

split_df.to_csv(OUTPUT_DIR / 'split_dataframe.csv', index=False)
print('Train:', len(train_df), 'Validation:', len(val_df), 'Test:', len(test_df))
display(split_df.groupby(['split', 'label']).size().unstack(fill_value=0))

# Quick sanity baseline: majority-class accuracy on validation.
val_label_counts = val_df['label'].value_counts(normalize=True).sort_values(ascending=False)
majority_label = val_label_counts.index[0]
majority_acc = float(val_label_counts.iloc[0])
print(f'Validation majority class: {majority_label} -> baseline accuracy: {majority_acc:.4f}')



In [ ]:
def export_split_folders(split_dataframe, output_root):
    output_root = Path(output_root)
    for _, row in split_dataframe.iterrows():
        src = Path(row['image_path'])
        dest = output_root / row['split'] / row['label'] / src.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            shutil.copy2(src, dest)

if EXPORT_SPLIT_FOLDERS:
    split_output = OUTPUT_DIR / 'data' / 'splits'
    export_split_folders(split_df, split_output)
    print('Split folders exported to:', split_output)
else:
    print('EXPORT_SPLIT_FOLDERS=False, split is stored as dataframe CSV only.')



In [ ]:
split_counts = split_df.groupby(['split', 'label']).size().reset_index(name='count')
plt.figure(figsize=(14, 5))
sns.barplot(data=split_counts, x='label', y='count', hue='split')
plt.title('Class distribution across train/validation/test')
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

distribution = pd.crosstab(split_df['label'], split_df['split'], normalize='columns')
distribution['train_test_abs_diff'] = (distribution['train'] - distribution['test']).abs()
display(distribution.round(4))

max_shift = distribution['train_test_abs_diff'].max()
if max_shift < 0.02:
    print(f'Distribution shift is small. Max train-test class proportion difference = {max_shift:.4f}')
else:
    print(f'Potential distribution shift. Max train-test class proportion difference = {max_shift:.4f}')



## Section 7: Image Preprocessing

Ảnh được resize về `224x224` để tạo batch tensor có cùng kích thước. Dữ liệu sau đó được đưa qua preprocessing kiểu `tf` của Keras Applications, tức quy đổi giá trị pixel về khoảng `[-1, 1]`, phù hợp với MobileNetV2 và ResNet50. Train set dùng augmentation nhẹ và MixUp nhẹ (`alpha=0.15`) để giảm overfit. Loss mặc định là focal loss; class weight có cap được hỗ trợ như một lựa chọn bổ sung để tránh NaN.



In [ ]:
label_to_id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_classes = len(label_to_id)

for frame in [train_df, val_df, test_df, split_df]:
    frame['label_id'] = frame['label'].map(label_to_id)

data_augmentation = tf.keras.Sequential([
    # Augmentation nhẹ: đủ để tăng khả năng tổng quát, không làm biến dạng mạnh phương tiện.
    tf.keras.layers.RandomFlip('horizontal', dtype='float32'),
    tf.keras.layers.RandomRotation(0.04, dtype='float32'),
    tf.keras.layers.RandomZoom(0.08, dtype='float32'),
    tf.keras.layers.RandomContrast(0.08, dtype='float32'),
], name='train_augmentation')

def mobilenet_preprocess(x):
    # MobileNetV2 expects pixels scaled from [0, 255] to [-1, 1].
    return tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)

def resnet50_preprocess(x):
    # ResNet50 ImageNet weights expect Keras ResNet preprocessing.
    return tf.keras.applications.resnet50.preprocess_input(x * 255.0)

def application_preprocess(x):
    return mobilenet_preprocess(x)

def load_and_preprocess(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def one_hot_label(image, label):
    return image, tf.one_hot(tf.cast(label, tf.int32), depth=num_classes, dtype=tf.float32)

def mixup_batch(images, labels):
    if not USE_MIXUP or MIXUP_ALPHA <= 0:
        return images, labels
    alpha = tf.cast(MIXUP_ALPHA, tf.float32)
    batch_size = tf.shape(images)[0]
    gamma1 = tf.random.gamma(shape=[batch_size], alpha=alpha, dtype=tf.float32)
    gamma2 = tf.random.gamma(shape=[batch_size], alpha=alpha, dtype=tf.float32)
    lam = gamma1 / (gamma1 + gamma2)
    lam_x = tf.reshape(lam, [batch_size, 1, 1, 1])
    lam_y = tf.reshape(lam, [batch_size, 1])
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = images * lam_x + tf.gather(images, indices) * (1.0 - lam_x)
    mixed_labels = labels * lam_y + tf.gather(labels, indices) * (1.0 - lam_y)
    return mixed_images, mixed_labels

class_weights = np.ones(num_classes, dtype=np.float32)
if USE_CLASS_WEIGHTS:
    class_counts = train_df['label_id'].value_counts().to_dict()
    total = len(train_df)
    for class_id in range(num_classes):
        count = max(1, int(class_counts.get(class_id, 0)))
        raw_weight = total / (num_classes * count)
        class_weights[class_id] = float(np.clip(raw_weight, CLASS_WEIGHT_FLOOR, CLASS_WEIGHT_CAP))

class_weight_tensor = tf.constant(class_weights, dtype=tf.float32)

def attach_sample_weight(images, labels):
    sample_weight = tf.reduce_sum(labels * class_weight_tensor, axis=-1)
    return images, labels, sample_weight

def make_dataset_with_preprocess(dataframe, preprocess_fn, training=False):
    paths = dataframe['image_path'].astype(str).values
    labels = dataframe['label_id'].astype('int32').values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)

    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.AUTO
    if training:
        options.deterministic = False
    ds = ds.with_options(options)

    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(tf.cast(x, tf.float32), training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(one_hot_label, num_parallel_calls=tf.data.AUTOTUNE)

    drop_remainder = bool(training and USE_MIXUP)
    ds = ds.batch(BATCH_SIZE, drop_remainder=drop_remainder)
    if training and USE_MIXUP:
        ds = ds.map(mixup_batch, num_parallel_calls=tf.data.AUTOTUNE)
    if USE_CLASS_WEIGHTS:
        ds = ds.map(attach_sample_weight, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def make_dataset(dataframe, training=False):
    return make_dataset_with_preprocess(dataframe, mobilenet_preprocess, training=training)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

# ResNet50 uses a different ImageNet preprocessing convention from MobileNetV2.
resnet_train_ds = make_dataset_with_preprocess(train_df, resnet50_preprocess, training=True)
resnet_val_ds = make_dataset_with_preprocess(val_df, resnet50_preprocess, training=False)
resnet_test_ds = make_dataset_with_preprocess(test_df, resnet50_preprocess, training=False)

print('Number of classes:', num_classes)
print(label_to_id)
print('Train batches per epoch:', int(np.ceil(len(train_df) / BATCH_SIZE)))
print('Validation batches:', int(np.ceil(len(val_df) / BATCH_SIZE)))
print('Test batches:', int(np.ceil(len(test_df) / BATCH_SIZE)))
if USE_CLASS_WEIGHTS:
    print('Class weights (capped):', {id_to_label[i]: float(class_weights[i]) for i in range(num_classes)})

# Store the label mapping for later inference or submission with the model.
label_map = {
    'label_to_id': label_to_id,
    'id_to_label': {str(k): v for k, v in id_to_label.items()},
}
(ARTIFACT_DIR / 'label_map.json').write_text(json.dumps(label_map, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved label map:', ARTIFACT_DIR / 'label_map.json')




In [ ]:
sample_row = train_df.sample(1, random_state=SEED).iloc[0]
original = Image.open(sample_row['image_path']).convert('RGB')
resized = original.resize(IMG_SIZE)
normalized = np.asarray(resized).astype('float32') / 255.0

# data_augmentation is applied to single images before batching in train_ds,
# so this visualization also uses a single image with shape (224, 224, 3).
augmented_input = tf.convert_to_tensor(normalized, dtype=tf.float32)
augmented = data_augmentation(augmented_input, training=True).numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(original); axes[0].set_title(f'Original {original.size}')
axes[1].imshow(resized); axes[1].set_title('Resized 224x224')
axes[2].hist(normalized.ravel(), bins=40); axes[2].set_title('Normalized pixel values')
axes[3].imshow(np.clip(augmented, 0, 1)); axes[3].set_title('Train augmentation example')
for ax in [axes[0], axes[1], axes[3]]:
    ax.axis('off')
plt.tight_layout()
plt.show()



## Section 8: Feature Extraction and Visualization

CNN sẽ học đặc trưng trực tiếp từ pixel ảnh đã resize/normalize. Để trực quan hóa trước khi train, ta dùng pixel feature đơn giản từ ảnh downsample `32x32` kết hợp PCA/t-SNE. Biểu đồ 2D giúp quan sát sơ bộ class nào dễ tách và class nào có thể chồng lấn.



In [ ]:
def build_visualization_features(dataframe, max_per_class=80):
    sampled = []
    for label in sorted(dataframe['label'].unique()):
        part = dataframe[dataframe['label'] == label]
        sampled.append(part.sample(min(max_per_class, len(part)), random_state=SEED))
    sample_df = pd.concat(sampled, ignore_index=True)

    features_raw = []
    features_norm = []
    labels = []
    for _, row in sample_df.iterrows():
        img = Image.open(row['image_path']).convert('RGB').resize((32, 32))
        arr = np.asarray(img).astype('float32')
        features_raw.append(arr.ravel())
        features_norm.append((arr / 255.0).ravel())
        labels.append(row['label'])
    return np.asarray(features_raw), np.asarray(features_norm), np.asarray(labels)

X_raw, X_norm, y_vis = build_visualization_features(df, max_per_class=80)
print('Raw pixel feature shape:', X_raw.shape)
print('Normalized pixel feature shape:', X_norm.shape)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X_raw.ravel(), bins=50, color='#756bb1')
axes[0].set_title('Pixel distribution before normalization')
axes[1].hist(X_norm.ravel(), bins=50, color='#238b45')
axes[1].set_title('Pixel distribution after normalization')
plt.tight_layout()
plt.show()



In [ ]:
X_scaled = StandardScaler().fit_transform(X_norm)
pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({'pc1': pca_coords[:, 0], 'pc2': pca_coords[:, 1], 'label': y_vis})

plt.figure(figsize=(9, 7))
sns.scatterplot(data=pca_df, x='pc1', y='pc2', hue='label', s=35, alpha=0.85)
plt.title('PCA visualization of normalized pixel features')
plt.tight_layout()
plt.show()

print('PCA explained variance ratio:', pca.explained_variance_ratio_)



In [ ]:
if len(X_scaled) >= 50:
    n_vis = min(len(X_scaled), 800)
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X_scaled), size=n_vis, replace=False)
    perplexity = min(30, max(5, n_vis // 10))
    tsne = TSNE(n_components=2, perplexity=perplexity, init='pca', learning_rate='auto', random_state=SEED)
    tsne_coords = tsne.fit_transform(X_scaled[idx])
    tsne_df = pd.DataFrame({'x': tsne_coords[:, 0], 'y': tsne_coords[:, 1], 'label': y_vis[idx]})
    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=tsne_df, x='x', y='y', hue='label', s=35, alpha=0.85)
    plt.title('t-SNE visualization of normalized pixel features')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough samples for t-SNE visualization.')



## Section 9: Model 1 - MobileNetV2

MobileNet dùng **depthwise separable convolution**: depthwise convolution học bộ lọc riêng cho từng channel, sau đó pointwise convolution `1x1` trộn thông tin giữa các channel. Cách này giảm số lượng tham số so với convolution thường.

Kiến trúc trong notebook dùng `tf.keras.applications.MobileNetV2`:

- Input size: `224x224x3`.
- `include_top=False` để bỏ classifier mặc định.
- Global average pooling + dropout + dense softmax cho classifier mới.
- Huấn luyện 2 giai đoạn:
  - Giai đoạn 1: freeze backbone, train classifier head.
  - Giai đoạn 2: unfreeze một phần backbone để tối ưu lại với learning rate nhỏ.

> Lưu ý khi chạy trên Kaggle T4x2: nếu bạn đổi kiến trúc hoặc sửa cell model rồi train lại, hãy **Restart Session** trước khi chạy lại từ đầu để tránh lỗi collective shape mismatch.



In [ ]:
def build_training_loss():
    if USE_FOCAL_LOSS:
        try:
            return tf.keras.losses.CategoricalFocalCrossentropy(
                gamma=FOCAL_GAMMA,
                from_logits=False,
                label_smoothing=FOCAL_LABEL_SMOOTHING,
            )
        except AttributeError:
            print('CategoricalFocalCrossentropy is unavailable. Fallback to CategoricalCrossentropy.')
    return tf.keras.losses.CategoricalCrossentropy(label_smoothing=FOCAL_LABEL_SMOOTHING)


def set_trainable_backbone(base_model, unfreeze_last_layers):
    base_model.trainable = True
    if unfreeze_last_layers is None or unfreeze_last_layers <= 0:
        for layer in base_model.layers:
            layer.trainable = False
        return base_model

    cutoff = max(0, len(base_model.layers) - int(unfreeze_last_layers))
    for i, layer in enumerate(base_model.layers):
        layer.trainable = i >= cutoff

    # Freeze BN for stable fine-tuning
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    return base_model


def merge_histories(*histories):
    frames = []
    for h in histories:
        if h is None:
            continue
        frames.append(pd.DataFrame(h.history))
    if not frames:
        return None
    merged = pd.concat(frames, ignore_index=True)
    obj = type('HistoryProxy', (), {})()
    obj.history = {k: merged[k].tolist() for k in merged.columns}
    return obj


def build_mobilenet_model(num_classes):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet' if USE_PRETRAINED else None,
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = tf.keras.layers.Dropout(MOBILENET_DROPOUT, name='dropout')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32', name='classifier')(x)

    model = tf.keras.Model(inputs, outputs, name='MobileNetV2_Model')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=WARMUP_LR, clipnorm=1.0),
        loss=build_training_loss(),
        metrics=['accuracy'],
    )
    return model, base_model


with strategy.scope():
    mobilenet_model, mobilenet_base = build_mobilenet_model(num_classes)

mobilenet_model.summary()




In [ ]:
mobilenet_best_path = MODEL_DIR / 'mobilenet_best.keras'
mobilenet_final_path = MODEL_DIR / 'mobilenet_final.keras'

callbacks_mobilenet = [
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(mobilenet_best_path),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR),
]

mobilenet_history = None
mobilenet_history_warmup = None
mobilenet_history_finetune = None
mobilenet_train_time = 0

if RUN_TRAINING:
    start = time.time()

    warmup_epochs = min(WARMUP_EPOCHS, EPOCHS)
    if warmup_epochs > 0:
        print(f'[MobileNet] Phase 1 warm-up: {warmup_epochs} epochs, lr={WARMUP_LR}')
        mobilenet_history_warmup = mobilenet_model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=warmup_epochs,
            callbacks=callbacks_mobilenet,
        )

    remaining_epochs = max(0, EPOCHS - warmup_epochs)
    if remaining_epochs > 0:
        print(f'[MobileNet] Phase 2 unfreeze last {MOBILENET_UNFREEZE_LAYERS} layers, lr={FINETUNE_LR}')
        with strategy.scope():
            mobilenet_base = set_trainable_backbone(mobilenet_base, MOBILENET_UNFREEZE_LAYERS)
            mobilenet_model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LR, clipnorm=1.0),
                loss=build_training_loss(),
                metrics=['accuracy'],
            )

        mobilenet_history_finetune = mobilenet_model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            initial_epoch=warmup_epochs,
            callbacks=callbacks_mobilenet,
        )

    mobilenet_history = merge_histories(mobilenet_history_warmup, mobilenet_history_finetune)
    mobilenet_train_time = time.time() - start

    mobilenet_model.save(str(mobilenet_final_path))
    if mobilenet_history is not None:
        pd.DataFrame(mobilenet_history.history).to_csv(ARTIFACT_DIR / 'mobilenet_history.csv', index=False)
    (ARTIFACT_DIR / 'mobilenet_architecture.json').write_text(mobilenet_model.to_json(), encoding='utf-8')

    print('Saved best MobileNet checkpoint:', mobilenet_best_path)
    print('Saved final MobileNet model:', mobilenet_final_path)
else:
    print('RUN_TRAINING=False, skip MobileNet training.')




In [ ]:
def safe_filename(name):
    return name.lower().replace(' ', '_').replace('-', '_')

def plot_training_history(history, title):
    if history is None:
        print(f'No history for {title}.')
        return
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(hist['loss'], label='train_loss')
    axes[0].plot(hist['val_loss'], label='val_loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    axes[1].plot(hist['accuracy'], label='train_accuracy')
    axes[1].plot(hist['val_accuracy'], label='val_accuracy')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].legend()
    plt.tight_layout()
    figure_path = FIGURE_DIR / f'{safe_filename(title)}_training_curves.png'
    fig.savefig(figure_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved training curve:', figure_path)

plot_training_history(mobilenet_history, 'MobileNetV2')




## Section 10: Model 2 - ResNet50

ResNet dùng **residual block** và **skip connection**. Thay vì buộc các lớp học trực tiếp một ánh xạ phức tạp, residual block học phần sai khác so với input. Skip connection giúp gradient truyền tốt hơn qua mạng sâu, giảm hiện tượng vanishing gradient.

Kiến trúc trong notebook dùng `tf.keras.applications.ResNet50`:

- Input size: `224x224x3`.
- `include_top=False` để bỏ classifier mặc định.
- Global average pooling + dropout + dense softmax cho classifier mới.
- Huấn luyện 2 giai đoạn:
  - Giai đoạn 1: freeze backbone, train classifier head.
  - Giai đoạn 2: unfreeze một phần backbone để tối ưu lại với learning rate nhỏ.



In [ ]:
def build_resnet50_model(num_classes):
    base_model = tf.keras.applications.ResNet50(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet' if USE_PRETRAINED else None,
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = tf.keras.layers.Dropout(RESNET_DROPOUT, name='dropout')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32', name='classifier')(x)

    model = tf.keras.Model(inputs, outputs, name='ResNet50_Model')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=WARMUP_LR, clipnorm=1.0),
        loss=build_training_loss(),
        metrics=['accuracy'],
    )
    return model, base_model

with strategy.scope():
    resnet_model, resnet_base = build_resnet50_model(num_classes)

resnet_model.summary()




In [ ]:
resnet_best_path = MODEL_DIR / 'resnet_best.keras'
resnet_final_path = MODEL_DIR / 'resnet_final.keras'

callbacks_resnet = [
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(resnet_best_path),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR),
]

resnet_history = None
resnet_history_warmup = None
resnet_history_finetune = None
resnet_train_time = 0
RESNET_FINETUNE_LR = 3e-5
RESNET_UNFREEZE_LAYERS = 80
if RUN_TRAINING:
    start = time.time()

    warmup_epochs = min(WARMUP_EPOCHS, EPOCHS)
    if warmup_epochs > 0:
        print(f'[ResNet50] Phase 1 warm-up: {warmup_epochs} epochs, lr={WARMUP_LR}')
        resnet_history_warmup = resnet_model.fit(
            resnet_train_ds,
            validation_data=resnet_val_ds,
            epochs=warmup_epochs,
            callbacks=callbacks_resnet,
        )

    remaining_epochs = max(0, EPOCHS - warmup_epochs)
    if remaining_epochs > 0:
        print(f'[ResNet50] Phase 2 unfreeze last {RESNET_UNFREEZE_LAYERS} layers, lr={RESNET_FINETUNE_LR}')
        with strategy.scope():
            resnet_base = set_trainable_backbone(resnet_base, RESNET_UNFREEZE_LAYERS)
            resnet_model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=RESNET_FINETUNE_LR, clipnorm=1.0),
                loss=build_training_loss(),
                metrics=['accuracy'],
            )

        resnet_history_finetune = resnet_model.fit(
            resnet_train_ds,
            validation_data=resnet_val_ds,
            epochs=EPOCHS,
            initial_epoch=warmup_epochs,
            callbacks=callbacks_resnet,
        )

    resnet_history = merge_histories(resnet_history_warmup, resnet_history_finetune)
    resnet_train_time = time.time() - start

    resnet_model.save(str(resnet_final_path))
    if resnet_history is not None:
        pd.DataFrame(resnet_history.history).to_csv(ARTIFACT_DIR / 'resnet_history.csv', index=False)
    (ARTIFACT_DIR / 'resnet_architecture.json').write_text(resnet_model.to_json(), encoding='utf-8')

    print('Saved best ResNet checkpoint:', resnet_best_path)
    print('Saved final ResNet model:', resnet_final_path)
else:
    print('RUN_TRAINING=False, skip ResNet training.')

plot_training_history(resnet_history, 'ResNet50')




## Section 11: Model Evaluation

Đánh giá hai mô hình trên test set bằng Accuracy, Precision, Recall, F1-score, confusion matrix và classification report. Bảng so sánh cũng ghi số tham số và thời gian train.

Sau khi train, notebook lưu các artifact quan trọng vào `ARTIFACT_DIR`, gồm best model checkpoint, final model, lịch sử train, label map, classification report, file dự đoán trên test set và các hình confusion matrix/training curve. Các file này nằm trong `/kaggle/working/training_artifacts` khi chạy trên Kaggle, có thể tải về ở phần Output của notebook.



In [ ]:
class_names = [id_to_label[i] for i in range(num_classes)]

def load_image_for_prediction(path, preprocess_fn):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    image = preprocess_fn(image)
    return image

def predict_dataframe(model, dataframe, preprocess_fn, batch_size=BATCH_SIZE):
    paths = dataframe['image_path'].astype(str).values
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(lambda path: load_image_for_prediction(path, preprocess_fn), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    probabilities = model.predict(ds, verbose=1)
    pred_ids = np.argmax(probabilities, axis=1)
    confidence = np.max(probabilities, axis=1)

    pred_df = dataframe[['image_path', 'label', 'label_id']].copy().reset_index(drop=True)
    pred_df['true_id'] = pred_df['label_id'].astype(int)
    pred_df['pred_id'] = pred_ids.astype(int)
    pred_df['pred_label'] = [id_to_label[int(i)] for i in pred_ids]
    pred_df['confidence'] = confidence
    pred_df['correct'] = pred_df['true_id'].values == pred_df['pred_id'].values
    return pred_df, probabilities

def load_best_model_if_exists(current_model, checkpoint_path, model_name):
    checkpoint_path = Path(checkpoint_path)
    if checkpoint_path.exists():
        print(f'Loading best checkpoint for {model_name}: {checkpoint_path}')
        with strategy.scope():
            return tf.keras.models.load_model(str(checkpoint_path), compile=False)
    print(f'Best checkpoint for {model_name} not found. Use current in-memory model.')
    return current_model

def evaluate_model(model, model_name, training_time, preprocess_fn):
    pred_df, probabilities = predict_dataframe(model, test_df, preprocess_fn)
    y_true = pred_df['true_id'].values
    y_pred = pred_df['pred_id'].values

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(num_classes)), average='macro', zero_division=0
    )

    print('\n' + '=' * 80)
    print(model_name)
    print('=' * 80)
    report_text = classification_report(
        y_true, y_pred, labels=list(range(num_classes)), target_names=class_names, zero_division=0
    )
    print(report_text)

    safe_name = safe_filename(model_name)
    report_dict = classification_report(
        y_true, y_pred, labels=list(range(num_classes)), target_names=class_names, zero_division=0, output_dict=True
    )
    pd.DataFrame(report_dict).transpose().to_csv(ARTIFACT_DIR / f'{safe_name}_classification_report.csv')
    pred_df.to_csv(ARTIFACT_DIR / f'{safe_name}_test_predictions.csv', index=False)
    np.save(ARTIFACT_DIR / f'{safe_name}_test_probabilities.npy', probabilities)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    fig = plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.xticks(rotation=35, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = FIGURE_DIR / f'{safe_filename(model_name)}_confusion_matrix.png'
    fig.savefig(cm_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved confusion matrix:', cm_path)

    return {
        'Model': model_name,
        'Test Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'Number of parameters': model.count_params(),
        'Training time (seconds)': training_time,
    }, cm, pred_df

evaluation_rows = []
confusion_matrices = {}
prediction_tables = {}

if RUN_TRAINING:
    mobilenet_eval_model = load_best_model_if_exists(mobilenet_model, mobilenet_best_path, 'MobileNetV2')
    resnet_eval_model = load_best_model_if_exists(resnet_model, resnet_best_path, 'ResNet50')

    row, cm, pred_df = evaluate_model(mobilenet_eval_model, 'MobileNetV2', mobilenet_train_time, mobilenet_preprocess)
    evaluation_rows.append(row)
    confusion_matrices['MobileNetV2'] = cm
    prediction_tables['MobileNetV2'] = pred_df

    row, cm, pred_df = evaluate_model(resnet_eval_model, 'ResNet50', resnet_train_time, resnet50_preprocess)
    evaluation_rows.append(row)
    confusion_matrices['ResNet50'] = cm
    prediction_tables['ResNet50'] = pred_df
else:
    print('RUN_TRAINING=False, evaluation is skipped.')

results_df = pd.DataFrame(evaluation_rows)
if not results_df.empty:
    display(results_df.round(4))
    results_df.to_csv(ARTIFACT_DIR / 'model_comparison.csv', index=False)
    print('Saved model comparison:', ARTIFACT_DIR / 'model_comparison.csv')





In [ ]:
if not results_df.empty:
    metric_cols = ['Test Accuracy', 'Precision', 'Recall', 'F1-score']
    plot_df = results_df.melt(id_vars='Model', value_vars=metric_cols, var_name='Metric', value_name='Score')
    fig = plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model')
    plt.ylim(0, 1)
    plt.title('MobileNetV2 vs ResNet50 metrics on test set')
    plt.tight_layout()
    metric_path = FIGURE_DIR / 'model_metric_comparison.png'
    fig.savefig(metric_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved metric comparison figure:', metric_path)

    params_time_df = results_df[['Model', 'Number of parameters', 'Training time (seconds)']].copy()
    display(params_time_df)
else:
    print('No evaluation results yet.')




### Visualize correct and wrong predictions

Cell dưới đây giúp kiểm tra chất lượng dự đoán bằng mắt thường:

- Với mỗi model, mỗi label lấy tối đa 5 ảnh trong test set mà model dự đoán đúng.
- Với các ảnh dự đoán sai, notebook gom theo label thật và hiển thị toàn bộ ảnh sai. Nếu số ảnh sai quá nhiều, có thể đặt `MAX_WRONG_IMAGES_PER_LABEL` thành một số cụ thể như `30` để xem nhanh hơn.
- Notebook cũng lưu CSV các ảnh đúng/sai vào `ARTIFACT_DIR` để tiện viết báo cáo hoặc lọc lỗi sau train.



In [ ]:
CORRECT_SAMPLES_PER_LABEL = 5
MAX_WRONG_IMAGES_PER_LABEL = None  # None = hiển thị toàn bộ ảnh sai của mỗi label
WRONG_IMAGES_PER_PAGE = 25

def render_prediction_grid(rows, title, max_cols=5):
    rows = rows.reset_index(drop=True)
    if rows.empty:
        print(f'No images to show: {title}')
        return

    cols = min(max_cols, len(rows))
    n_rows = math.ceil(len(rows) / cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 3.2, n_rows * 3.6))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, rows.iterrows()):
        try:
            image = Image.open(row['image_path']).convert('RGB')
            ax.imshow(image)
            ax.set_title(
                f"true: {row['label']}\npred: {row['pred_label']}\nconf: {row['confidence']:.2f}",
                fontsize=9,
            )
        except Exception as exc:
            ax.text(0.5, 0.5, f'Cannot open image\n{exc}', ha='center', va='center')
        ax.axis('off')

    for ax in axes[len(rows):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

def select_correct_examples(pred_df, samples_per_label=5):
    selected = []
    missing_labels = []
    for label in class_names:
        subset = pred_df[(pred_df['label'] == label) & (pred_df['correct'])]
        if subset.empty:
            missing_labels.append(label)
            continue
        selected.append(subset.sample(n=min(samples_per_label, len(subset)), random_state=SEED))

    if missing_labels:
        print('Labels without correct predictions:', ', '.join(missing_labels))
    if not selected:
        return pd.DataFrame(columns=pred_df.columns)
    return pd.concat(selected, ignore_index=True)

def visualize_correct_predictions(pred_df, model_name):
    selected = select_correct_examples(pred_df, CORRECT_SAMPLES_PER_LABEL)
    safe_name = safe_filename(model_name)
    selected.to_csv(ARTIFACT_DIR / f'{safe_name}_correct_examples.csv', index=False)
    print(f'{model_name}: selected {len(selected)} correct examples.')
    render_prediction_grid(selected, f'{model_name} - 5 correct examples per label')

def visualize_wrong_predictions(pred_df, model_name):
    wrong = pred_df[~pred_df['correct']].copy()
    safe_name = safe_filename(model_name)
    wrong.to_csv(ARTIFACT_DIR / f'{safe_name}_wrong_predictions.csv', index=False)

    if wrong.empty:
        print(f'{model_name}: no wrong predictions on test set.')
        return

    print(f'{model_name}: {len(wrong)} wrong predictions.')
    wrong_summary = wrong.groupby(['label', 'pred_label']).size().reset_index(name='count')
    display(wrong_summary.sort_values(['label', 'count'], ascending=[True, False]))

    for label in class_names:
        subset = wrong[wrong['label'] == label].sort_values('confidence', ascending=False)
        if subset.empty:
            continue
        if MAX_WRONG_IMAGES_PER_LABEL is not None:
            subset = subset.head(MAX_WRONG_IMAGES_PER_LABEL)

        for start in range(0, len(subset), WRONG_IMAGES_PER_PAGE):
            page = subset.iloc[start:start + WRONG_IMAGES_PER_PAGE]
            page_no = start // WRONG_IMAGES_PER_PAGE + 1
            render_prediction_grid(page, f'{model_name} - Wrong predictions for true label: {label} - page {page_no}')

if prediction_tables:
    for model_name, pred_df in prediction_tables.items():
        print('\n' + '=' * 80)
        print(model_name)
        print('=' * 80)
        visualize_correct_predictions(pred_df, model_name)
        visualize_wrong_predictions(pred_df, model_name)
else:
    print('No prediction tables yet. Run training and evaluation first.')



### Saved artifacts

Sau khi notebook chạy xong trên Kaggle, các file quan trọng nằm trong `/kaggle/working/training_artifacts`. Có thể tải toàn bộ output của notebook hoặc tải từng file model `.keras`, CSV report và hình `.png`.



In [ ]:
saved_artifacts = sorted([p for p in ARTIFACT_DIR.rglob('*') if p.is_file()])
if saved_artifacts:
    artifact_df = pd.DataFrame({
        'artifact_path': [str(p) for p in saved_artifacts],
        'size_mb': [p.stat().st_size / (1024 * 1024) for p in saved_artifacts],
    })
    display(artifact_df.round({'size_mb': 3}))
else:
    print('No artifacts saved yet.')



## Section 12: Result Discussion

Sau khi chạy training/evaluation, hãy dựa vào bảng metric và confusion matrix để nhận xét:

- Mô hình có `F1-score` và `Test Accuracy` cao hơn thường là mô hình tổng quát tốt hơn trên test set.
- MobileNetV2 thường nhẹ hơn vì dùng depthwise separable convolution và inverted residual block, phù hợp khi cần tốc độ và ít tham số.
- ResNet50 có skip connection nên học ổn định hơn CNN thuần, năng lực biểu diễn mạnh hơn nhưng thời gian train thường cao hơn.
- Các class dễ nhầm lẫn thường là `car` và `bus`, `minibus` và `bus`, `truck` và `bus`, `motorcycle` và `bicycle`.
- Chất lượng dữ liệu crawl và mức cân bằng class ảnh hưởng trực tiếp đến khả năng đạt accuracy >= 85%.



In [ ]:
if not results_df.empty:
    best_row = results_df.sort_values('F1-score', ascending=False).iloc[0]
    print(f"Best model by macro F1-score: {best_row['Model']}")
    print(f"Test Accuracy: {best_row['Test Accuracy']:.4f}")
    print(f"F1-score: {best_row['F1-score']:.4f}")
    if best_row['Test Accuracy'] >= 0.85:
        print('The model reaches the target accuracy >= 85%.')
    else:
        print('The model has not reached 85% accuracy yet. Improve data quality, class balance, augmentation, epochs, or learning rate.')
else:
    print('Run training first to generate automatic result discussion.')



## Section 13: Conclusion

Notebook đã trình bày đầy đủ quy trình cho đề tài:

1. Crawl data theo class phương tiện giao thông.
2. Clean data: xóa ảnh lỗi, ảnh quá nhỏ, chuẩn hóa RGB/JPEG, đổi tên thống nhất.
3. Deduplicate data bằng pHash/dHash và Hamming distance.
4. Chia train/validation/test bằng stratified split.
5. Preprocess ảnh: resize `224x224`, normalize theo preprocessing của Keras Applications, augmentation cho train set.
6. Trực quan hóa đặc trưng bằng PCA/t-SNE.
7. Train MobileNetV2.
8. Train ResNet50.
9. Đánh giá bằng Accuracy, Precision, Recall, F1-score, classification report và confusion matrix.
10. So sánh hai mô hình bằng bảng số liệu và biểu đồ.
11. Lưu artifact sau train: best model, final model, history CSV, prediction CSV, confusion matrix.

Khi muốn cải thiện thêm kết quả, có thể mở rộng dữ liệu, tăng chất lượng lọc thủ công, tinh chỉnh augmentation và learning rate schedule.

